This is a brief demonstration of the workflow for the quantum protein folding problem on the FCC lattice. In the following sections, we present two key methods for building and solving the FCC Hamiltonian: polynomial fitting and the Variational Quantum Eigensolver with Constraints (VQEC) based on the Lagrangian dual method.

In [ ]:
# Autoload modules
%load_ext autoreload
%autoreload 2

In [ ]:
# Necessary imports
from fcc import MiyazawaJerniganInteraction, Peptide, ProteinFoldingProblem, PenaltyParameters, ProteinSolver, ProteinFoldingResult, ProteinShapeDecoder, build_turn_only_fcc_model
import fcc
from vqe import ChanceConstrainedVQEC
from qiskit.circuit.library import RealAmplitudes
from qiskit_aer.primitives import SamplerV2 as Sampler
import matplotlib.pyplot as plt
import ray
import psutil
import numpy as np
from time import time

In [ ]:
# Initialize Ray for parallel processing
num_workers = psutil.cpu_count(logical=False)
print(f"Number of workers: {num_workers}")
ray.init(
    num_cpus=num_workers,
    ignore_reinit_error=True,
    log_to_driver=False,
    runtime_env={
        "py_modules": [fcc],
    },
)

## PolyFit

In [ ]:
protein_seq = "GNLVS"  # Define the amino acid sequence of the protein of interest
penalty_back = 100.0  # Backtracking penalty
penalty_redun = 100.0  # Penalty for the 4 redundant bitstrings that do not encode any of the 12 FCC directions
penalty_olap = 100.0  # Penalty for overlapping conformations

# Build the peptide object
peptide = Peptide(protein_seq)
# Set up the interaction and penalty terms
mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file="mj_matrix")
penalty_terms = PenaltyParameters(
    penalty_back=penalty_back, penalty_redun=penalty_redun, penalty_olap=penalty_olap
)
# Build the protein folding problem
pf_problem = ProteinFoldingProblem(
    peptide=peptide, interaction=mj_interaction, penalty_parameters=penalty_terms
)
# Get the Hamiltonian
qubit_op = pf_problem.qubit_op(r2_threshold=1.0, chunk=20)
print(f"Number of qubits: {qubit_op.num_qubits}")
print(f"Number of terms in the Hamiltonian: {qubit_op.size}")

In [ ]:
# Choose the ansatz for the VQE
ansatz = RealAmplitudes(qubit_op.num_qubits, reps=1, entanglement="linear").decompose()
ansatz.measure_all()  # Add measurement to the ansatz
# Set up the sampler
shots = 10_000
sampler = Sampler(default_shots=shots)
# Set up the solver
optimizer = "COBYLA"
max_iter = 200
num_batches = num_workers  # Parallelize the energy evaluations
protein_solver = ProteinSolver(
    ansatz=ansatz,
    hamiltonian=qubit_op,
    sampler=sampler,
    parallelizer="ray",  # Use Ray for parallel processing
)
# Run the VQE
start_time = time()
polyfit_result = protein_solver.train(
    optimizer=optimizer,
    maxiter=max_iter,
    num_batches=num_batches,
)
end_time = time()
print(f"VQE completed in {end_time - start_time:.2f} seconds")
print(f"VQE result keys: {polyfit_result.keys()}")

In [ ]:
polyfit_result["top_solutions"][:5]

In [ ]:
# Plot the top 5 solutions
for i, (bitstring, energy) in enumerate(polyfit_result["top_solutions"][:5]):
    pf_result = ProteinFoldingResult(
        peptide=peptide,
        unused_qubits=pf_problem.unused_qubits,
        solution_bitstring=bitstring,
    )
    fig = pf_result.get_figure(
        title=f"Best solution {i+1}: {bitstring} (Energy: {energy:.2f})"
    )

## Turn-only chance-constrained VQEC

The revised VQEC path uses only the compact FCC turn register. Each sampled bitstring is decoded and scored with $H_{back} + H_{redun} + \sum_{m,n} \epsilon_{mn} \mathbf{1}[D_{mn}=2]$. One explicit chance constraint $\Pr[D_{mn}=0]-\delta_{mn} \leq 0$ is imposed for each residue pair separated by at least three sequence positions. No contact ancillas or optimizer-side postselection are used.

In [ ]:
protein_seq = "GNLVS"  # Define the amino acid sequence of interest
penalty_back = 100.0  # Backtracking penalty
penalty_redun = 100.0  # Penalty for unused four-bit turn codes

# Build the peptide object
peptide = Peptide(protein_seq)
# Set up the interaction and penalty terms
mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file="mj_matrix")
penalty_terms = PenaltyParameters(
    penalty_back=penalty_back, penalty_redun=penalty_redun
)
# Build the compact sample-scored model. Geometry maps and contact ancillas
# from the historical symbolic path are not constructed.
turn_only_model = build_turn_only_fcc_model(
    peptide, interaction=mj_interaction, penalty_parameters=penalty_terms
)
delta_mn = np.full(turn_only_model.constraint_count, 0.01)
print(f"Number of turn qubits: {turn_only_model.num_qubits}")
print("Chance-constraint limits:")
for pair, limit in zip(turn_only_model.constrained_pairs, delta_mn):
    print(f"  {pair}: {limit}")

In [ ]:
# Use an unmeasured compact-register ansatz. The solver adds measurement
# to its own copy and evaluates diagonal primitives from every sample.
ansatz = RealAmplitudes(
    turn_only_model.num_qubits, reps=2, entanglement="linear"
).decompose()
shots = 10_000
sampler = Sampler(default_shots=shots, seed=7)

# Hyperparameters for the VQEC optimization
initial_params = np.random.default_rng(7).uniform(0, 2 * np.pi, ansatz.num_parameters)
init_dual_vars = np.zeros(turn_only_model.constraint_count)
perturb_step = 0.05
gamma = 0.1
max_iter = 100

vqec_solver = ChanceConstrainedVQEC(
    turn_only_model,
    ansatz,
    sampler,
    constraint_limits=delta_mn,
    shots=shots,
)
vqec_result = vqec_solver.optimize_primal_dual(
    initial_params=initial_params,
    initial_dual_vars=init_dual_vars,
    primal_perturb_step=perturb_step,
    dual_perturb_step=perturb_step,
    gamma=gamma,
    auto_update_step=False,
    max_iter=max_iter,
)

In [ ]:
vqec_result.keys()

In [ ]:
# Plot the energy and constraint expectations
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(vqec_result["energy_history"], label="Energy", marker=".")
ax[0].set_xlabel("Iteration")
ax[0].set_ylabel("Energy expectation")

for i, pair in enumerate(turn_only_model.constrained_pairs):
    ax[1].plot(
        np.array(vqec_result["constraints_history"])[:, i],
        label=f"Pair {pair}",
        marker=".",
    )
ax[1].set_xlabel("Iteration")
ax[1].set_ylabel(r"Chance residual $\Pr[D_{mn}=0]-\delta_{mn}$")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sample the optimized circuit through the same unfiltered solver path.
counts = vqec_solver.sample_counts(vqec_result["optimal_primal_vars"])[0]

# Sort the binary strings by their counts
sorted_quasi_dist = dict(sorted(counts.items(), key=lambda item: item[1], reverse=True))
# Score the most frequent samples directly from their decoded geometries.
bitstring_energies = {}
for i, (sample, count) in enumerate(sorted_quasi_dist.items()):
    if i >= 5:
        break
    evaluation = turn_only_model.evaluate_bitstring(sample)
    bitstring_energies[sample] = evaluation.objective
    print(
        f"Sample {i}: {sample}, probability={count / shots:.4f}, "
        f"objective={evaluation.objective:.4f}, "
        f"overlaps={evaluation.overlap_indicators.astype(int).tolist()}"
    )

In [ ]:
# Invalid turn codes remain part of optimization statistics; only the
# optional structure visualization below requires a physical encoding.
for i, (sample, energy) in enumerate(bitstring_energies.items()):
    if not turn_only_model.evaluate_bitstring(sample).physical_encoding:
        print(f"Skipping visualization of nonphysical turn code: {sample}")
        continue
    pf_result = ProteinFoldingResult(
        peptide=peptide,
        unused_qubits=[],
        solution_bitstring=sample,
    )
    fig = pf_result.get_figure(
        title=f"Most frequent solution {i+1}: {sample} (Energy: {energy:.2f})"
    )
    plt.show()

## Comparison to classical exhaustive search results

In [ ]:
from fcc.classical_utils import load_top_cls_solns

file_name = "topobj_GNLVS.txt"
top_cls_solutions = load_top_cls_solns(file_name)
top_cls_solutions[:5]

In [ ]:
# Example: Translate the top polyfit solutions to turn sequences
top_polyfit_solutions = polyfit_result["top_solutions"][:5]
print(top_polyfit_solutions)

polyfit_turns = []
for i, (bitstring, energy) in enumerate(top_polyfit_solutions):
    pf_decoder = ProteinShapeDecoder(
        peptide=peptide,
        solution_bitstring=bitstring,
    )
    polyfit_turns.append([pf_decoder.turn_sequence, energy])

polyfit_turns